<a href="https://colab.research.google.com/github/pk-sk-25/studyspot/blob/main/ABMIS_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#v final Bite Mark Analysis without RESNET
#vfinal - non-resnet
import cv2
import numpy as np
import os
import datetime
import matplotlib.pyplot as plt

def analyze_query_and_subject(query_image_path, subjects_folder, results_folder, display_results=True):
    os.makedirs(results_folder, exist_ok=True)

    # Process the query image
    query_save_folder = os.path.join(results_folder, "Query_Debug")
    print("[INFO] Processing query image...")
    query_data = process_image(
        image_path=query_image_path,
        save_steps=True,
        save_folder=query_save_folder,
        prefix="Query",
        min_area=1,  # Reduced to allow smaller contours
        max_area=100000,  # Increased to handle large objects
        circularity_threshold=0.02  # Relaxed to capture irregular shapes
    )

    # Process all subject images
    best_match = None
    best_similarity = -1
    best_subject_data = None

    for subject_folder in sorted(os.listdir(subjects_folder)):
        subject_path = os.path.join(subjects_folder, subject_folder)
        if not os.path.isdir(subject_path):
            continue

        for subject_image_name in os.listdir(subject_path):
            subject_image_path = os.path.join(subject_path, subject_image_name)
            if not os.path.isfile(subject_image_path):
                continue

            subject_save_folder = os.path.join(results_folder, f"{subject_folder}_Debug")
            print(f"[INFO] Processing subject image: {subject_image_path}")
            subject_data = process_image(
                image_path=subject_image_path,
                save_steps=True,
                save_folder=subject_save_folder,
                prefix=subject_folder,
                min_area=1,
                max_area=100000,
                circularity_threshold=0.02
            )

            # Compare similarity based on contour count
            query_contour_count = len(query_data["contours"])
            subject_contour_count = len(subject_data["contours"])
            similarity = min(query_contour_count, subject_contour_count) / max(query_contour_count, subject_contour_count)

            if similarity > best_similarity:
                best_similarity = similarity
                best_match = subject_image_path
                best_subject_data = subject_data

    # Display results
    if display_results and best_subject_data:
        display_best_match(query_data, best_subject_data, query_image_path, best_match)

    print(f"[INFO] Best match: {best_match} with similarity: {best_similarity:.2f}")
    print("[INFO] Processing complete. Check results folder for debug images.")

def process_image(image_path, save_steps, save_folder, prefix, min_area=1, max_area=100000, circularity_threshold=0.02):
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Cannot load image: {image_path}")

    os.makedirs(save_folder, exist_ok=True)
    debug_images = {}

    # --- Step 1: Blue Putty Isolation ---
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    lower_blue = np.array([70, 30, 30])  # Expanded range
    upper_blue = np.array([150, 255, 255])  # Broader range
    step1 = cv2.inRange(hsv, lower_blue, upper_blue)
    debug_images["Step1"] = step1

    # --- Step 2: Close Mask (Fill Holes) ---
    kernel_large = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (20, 20))  # Increased kernel size
    step2 = cv2.morphologyEx(step1, cv2.MORPH_CLOSE, kernel_large)
    debug_images["Step2"] = step2

    # --- Step 3: ROI Extraction ---
    step3 = cv2.bitwise_and(img, img, mask=step2)
    debug_images["Step3"] = step3

    # --- Step 4: Grayscale Conversion ---
    step4 = cv2.cvtColor(step3, cv2.COLOR_BGR2GRAY)
    debug_images["Step4"] = step4

    # --- Step 5: Adaptive Thresholding ---
    step5 = cv2.adaptiveThreshold(step4, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 31, 2)
    debug_images["Step5"] = step5

    # --- Step 6: Morphological Closing ---
    kernel_small = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    step6 = cv2.morphologyEx(step5, cv2.MORPH_CLOSE, kernel_small)
    debug_images["Step6"] = step6

    # --- Step 7: Contour Detection ---
    contours, debug_contours_image = find_teeth_contours(
        step6,
        min_area,
        max_area,
        circularity_threshold,
        original=img  # Pass original image so we can draw on top
    )
    debug_images["Step7"] = debug_contours_image

    # Save debug images
    if save_steps:
        for step_name, image in debug_images.items():
            cv2.imwrite(os.path.join(save_folder, f"{prefix}_{step_name}.jpeg"), image)

    return {
        "original": img,
        "step6": step6,
        "step7": debug_contours_image,
        "contours": contours
    }

def find_teeth_contours(mask, min_area, max_area, circularity_threshold, original=None):
    """
    Optimized function to find and filter contours in a binary mask,
    and optionally draw them on the original image.
    """
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    filtered_contours = []

    if original is not None:
        debug_image = original.copy()
    else:
        debug_image = np.zeros_like(mask)

    overlay = debug_image.copy()

    for contour in contours:
        area = cv2.contourArea(contour)
        if area < min_area or area > max_area:
            continue

        perimeter = cv2.arcLength(contour, True)
        if perimeter == 0:
            continue

        circularity = (4 * np.pi * area) / (perimeter ** 2)
        if circularity >= circularity_threshold:
            filtered_contours.append(contour)

    if filtered_contours:
        cv2.drawContours(overlay, filtered_contours, -1, (0, 255, 0), -1)
        alpha = 0.4
        debug_image = cv2.addWeighted(overlay, alpha, debug_image, 1 - alpha, 0)
        cv2.drawContours(debug_image, filtered_contours, -1, (0, 0, 255), 2)

    print(f"[DEBUG] Total valid contours: {len(filtered_contours)}")
    return filtered_contours, debug_image

def display_best_match(query_data, best_data, query_path, best_match_path):
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))

    # Query images
    axes[0, 0].imshow(cv2.cvtColor(query_data["original"], cv2.COLOR_BGR2RGB))
    axes[0, 0].set_title("Query - Original")
    axes[0, 1].imshow(query_data["step6"], cmap="gray")
    axes[0, 1].set_title("Query - Step6")
    axes[0, 2].imshow(query_data["step7"], cmap="gray")
    axes[0, 2].set_title("Query - Step7 (Contours)")

    # Best match images
    axes[1, 0].imshow(cv2.cvtColor(best_data["original"], cv2.COLOR_BGR2RGB))
    axes[1, 0].set_title("BestMatch - Original")
    axes[1, 1].imshow(best_data["step6"], cmap="gray")
    axes[1, 1].set_title("BestMatch - Step6")
    axes[1, 2].imshow(best_data["step7"], cmap="gray")
    axes[1, 2].set_title("BestMatch - Step7 (Contours)")

    for ax in axes.ravel():
        ax.axis("off")

    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    query_image_path = "/content/drive/MyDrive/Query/Test06.jpg"
    subjects_folder = "/content/drive/MyDrive/forensics_bite_marks/"
    results_folder = os.path.join(
        "/content/drive/MyDrive/forensics_bite_marks/results",
        f"BiteMarkAnalysis_v21_{datetime.datetime.now().strftime('%Y%m%d_%H%M')}"
    )

    analyze_query_and_subject(query_image_path, subjects_folder, results_folder)

[INFO] Processing query image...


ValueError: Cannot load image: /content/drive/MyDrive/Query/Test06.jpg

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from torchvision import models, transforms
import torch
import torch.nn as nn
from PIL import Image

# =====================================================
# 0) Global Settings
# =====================================================
BASE_FOLDER = "/content/drive/MyDrive/forensics_bite_marks/"
QUERY_PATH = "/content/drive/MyDrive/Query/Test06.jpg"  # Update the query path here
CACHE_FOLDER = "/content/drive/MyDrive/cache/"
DEBUG_FOLDER = "/content/drive/MyDrive/debug/"
os.makedirs(CACHE_FOLDER, exist_ok=True)
os.makedirs(DEBUG_FOLDER, exist_ok=True)

# HSV for blue putty (adjust as needed for your dataset)
HSV_LOWER = np.array([70, 20, 20])
HSV_UPPER = np.array([150, 255, 255])

# Contour filtering
MIN_AREA = 5
MAX_AREA = 300000  # Increased to handle large impressions

# =====================================================
# 1) ResNet Feature Extractor
# =====================================================
class ResNetFeatureExtractor(nn.Module):
    def __init__(self):
        super(ResNetFeatureExtractor, self).__init__()
        self.resnet = models.resnet50(weights="ResNet50_Weights.IMAGENET1K_V1")
        self.resnet = nn.Sequential(*list(self.resnet.children())[:-1])  # Remove classification layer

    def forward(self, x):
        x = self.resnet(x)
        return x.view(x.size(0), -1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
resnet_model = ResNetFeatureExtractor().to(device)
resnet_model.eval()

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def load_or_extract_resnet_features(image_path, cache_folder):
    cache_file = os.path.join(cache_folder, f"{os.path.basename(image_path)}.npy")
    if os.path.exists(cache_file):
        return np.load(cache_file)
    try:
        image = Image.open(image_path).convert("RGB")
        image_tensor = transform(image).unsqueeze(0).to(device)
        with torch.no_grad():
            features = resnet_model(image_tensor).cpu().numpy().flatten()
        features = features / (np.linalg.norm(features) + 1e-12)  # Normalize
        np.save(cache_file, features)
        return features
    except Exception as e:
        print(f"[ERROR] ResNet feature extraction failed for {image_path}: {e}")
        return None

# =====================================================
# 2) Contour-Based Preprocessing
# =====================================================
def preprocess_image(image_path, debug_folder=None):
    image = cv2.imread(image_path)
    if image is None:
        print(f"[ERROR] Unable to read image: {image_path}")
        return [], {}

    if debug_folder:
        os.makedirs(debug_folder, exist_ok=True)

    # Step 1: HSV Masking
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, HSV_LOWER, HSV_UPPER)
    if debug_folder:
        cv2.imwrite(os.path.join(debug_folder, "hsv_mask.png"), mask)

    # Step 2: Morphological Closing
    kernel = np.ones((5, 5), np.uint8)
    closed_mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=1)
    if debug_folder:
        cv2.imwrite(os.path.join(debug_folder, "closed_mask.png"), closed_mask)

    # Step 3: ROI Extraction
    roi = cv2.bitwise_and(image, image, mask=closed_mask)
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    if debug_folder:
        cv2.imwrite(os.path.join(debug_folder, "gray_roi.png"), gray)

    # Step 4: Thresholding
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    if debug_folder:
        cv2.imwrite(os.path.join(debug_folder, "binary_mask.png"), binary)

    # Step 5: Morphological Operations
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=1)
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel, iterations=1)
    if debug_folder:
        cv2.imwrite(os.path.join(debug_folder, "final_mask.png"), binary)

    # Step 6: Contour Detection
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    valid_contours = [c for c in contours if MIN_AREA < cv2.contourArea(c) < MAX_AREA]

    print(f"[DEBUG] {os.path.basename(image_path)}: total={len(contours)}, valid={len(valid_contours)}")

    return valid_contours, {
        "hsv_mask": mask,
        "binary_mask": binary,
        "final_mask": binary,
        "gray_roi": gray
    }

def extract_geometric_features(contours):
    if not contours:
        return np.zeros(10, dtype=np.float32)

    centroids = []
    for c in contours:
        M = cv2.moments(c)
        if M["m00"] != 0:
            centroids.append((M["m10"] / M["m00"], M["m01"] / M["m00"]))

    distances = []
    for i in range(len(centroids)):
        for j in range(i + 1, len(centroids)):
            dist = np.linalg.norm(np.array(centroids[i]) - np.array(centroids[j]))
            distances.append(dist)

    if len(distances) > 10:
        distances = distances[:10]
    else:
        distances = np.pad(distances, (0, 10 - len(distances)), constant_values=0)

    return np.array(distances, dtype=np.float32)

# =====================================================
# 3) Training Feature Extraction
# =====================================================
def extract_training_features(base_folder, cache_folder):
    features, labels = [], []
    for subject in sorted(os.listdir(base_folder)):
        subject_path = os.path.join(base_folder, subject)
        if not os.path.isdir(subject_path):
            continue
        for image_name in os.listdir(subject_path):
            image_path = os.path.join(subject_path, image_name)
            try:
                contours, _ = preprocess_image(image_path)
                geom_features = extract_geometric_features(contours)
                cnn_features = load_or_extract_resnet_features(image_path, cache_folder)
                if cnn_features is None:
                    continue
                combined_features = np.hstack([cnn_features, geom_features])
                features.append(combined_features)
                labels.append((subject, image_path))
            except Exception as e:
                print(f"[ERROR] Failed to process {image_name}: {e}")
    print(f"[INFO] Total training images processed: {len(features)}")
    return np.array(features), labels

# =====================================================
# 4) Query Matching and Visualization
# =====================================================
def match_and_visualize(query_path, training_features, training_labels, cache_folder, debug_folder, top_k=5):
    contours, debug_info = preprocess_image(query_path, debug_folder=debug_folder)

    if len(contours) == 0:
        print("[WARNING] No valid contours in query. Geom features = 0.")
        geom_features = np.zeros(10, dtype=np.float32)
    else:
        geom_features = extract_geometric_features(contours)

    cnn_features = load_or_extract_resnet_features(query_path, cache_folder)
    if cnn_features is None:
        print("[ERROR] No CNN features for query. Aborting.")
        return []

    query_features = np.hstack([cnn_features, geom_features]).reshape(1, -1)

    similarities = cosine_similarity(query_features, training_features)[0]
    top_matches = sorted(zip(training_labels, similarities), key=lambda x: -x[1])[:top_k]

    fig, axes = plt.subplots(6, top_k + 1, figsize=(15, 12))
    axes[0, 0].imshow(cv2.cvtColor(cv2.imread(query_path), cv2.COLOR_BGR2RGB))
    axes[0, 0].set_title("Query")
    axes[0, 0].axis("off")
    for rank, ((subject, image_path), sim) in enumerate(top_matches, start=1):
        axes[0, rank].imshow(cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB))
        axes[0, rank].set_title(f"Match {rank}: {subject}\nSim: {sim:.2f}")
        axes[0, rank].axis("off")
    plt.tight_layout()
    plt.show()

# =====================================================
# 5) Main
# =====================================================
if __name__ == "__main__":
    print("[INFO] Extracting training features...")
    train_features, train_labels = extract_training_features(BASE_FOLDER, CACHE_FOLDER)

    if len(train_features) == 0:
        print("[ERROR] No training features found. Aborting.")
    else:
        print("[INFO] Matching query...")
        match_and_visualize(QUERY_PATH, train_features, train_labels, CACHE_FOLDER, DEBUG_FOLDER)